In [0]:
# ─── Stage 3 — Gold Imperative v2 ────────────────────────────────────────────
# Thin orchestration notebook — logic lives in src/market_pulse/gold/
# ─────────────────────────────────────────────────────────────────────────────

import sys
for key in list(sys.modules.keys()):
    if 'market_pulse' in key:
        del sys.modules[key]

sys.path.insert(0, "/Workspace/Repos/martalimas@gmail.com/market-pulse-pipeline/src")

from market_pulse.gold import (
    read_silver_fact_prices,
    build_daily_summary,
    build_volume_analysis,
    build_moving_averages,
    build_volatility,
    build_stock_comparison
)
from market_pulse.utils import write_delta_table
from market_pulse.config import (
    GOLD_DAILY_SUMMARY_PATH,
    GOLD_VOLUME_ANALYSIS_PATH,
    GOLD_MOVING_AVERAGES_PATH,
    GOLD_VOLATILITY_PATH,
    GOLD_STOCK_COMPARISON_PATH
)

print("✅ Módulos importados de src/market_pulse/gold/")

In [0]:
# ─── Gold Pipeline ────────────────────────────────────────────────────────────
from market_pulse.logger import get_logger
from market_pulse.config import PIPELINE_LOG_PATH

logger = get_logger(__name__, spark=spark, log_table_path=PIPELINE_LOG_PATH)
logger.info("gold_pipeline_started")


# 1. Read Silver
df_fact = read_silver_fact_prices(spark, logger=logger)

# 2. Build and write all metrics
write_delta_table(
    build_daily_summary(df_fact, logger=logger),
    GOLD_DAILY_SUMMARY_PATH, "daily_summary", mode="overwrite"
)

write_delta_table(
    build_volume_analysis(spark, logger=logger),
    GOLD_VOLUME_ANALYSIS_PATH, "volume_analysis", mode="overwrite"
)

write_delta_table(
    build_moving_averages(spark, logger=logger),
    GOLD_MOVING_AVERAGES_PATH, "moving_averages", mode="overwrite"
)

write_delta_table(
    build_volatility(spark, logger=logger),
    GOLD_VOLATILITY_PATH, "volatility", mode="overwrite"
)

write_delta_table(
    build_stock_comparison(spark, logger=logger),
    GOLD_STOCK_COMPARISON_PATH, "stock_comparison", mode="overwrite"
)

print("✅ Gold pipeline complete")

In [0]:
#spark.read.format("delta").load(PIPELINE_LOG_PATH).orderBy("timestamp").display()